In [0]:
%sql
-- Create database autoloader in data_dev catalog
-- create database if not exists data_dev.autoloader

In [0]:
%sql
-- Create a volume Bronze in this database
-- create volume data_dev.autoloader.bronze

-- create 2 directory raw & destination from here using UI. In destination create 2 more directory/folder data & checkpoint. uploaded batch1.json in raw folder try & try to get it one time trigger to destination data folder.

**Autoloader**

In [0]:
# YT: https://www.youtube.com/watch?v=IkdoVaEq9jQ
# option("cloudFiles.schemaLocation", "/Volumes/data_dev/autoloader/bronze/destination/checkpoint/"): schema location. Where we want to store the schema in destination folder.
# option("cloudFiles.schemaEvolutionMode", "addNewColumns"): addNewColumns: Add new columns to destination
# option("cloudFiles.schemaEvolutionMode", "rescue"): rescue: Rescue files with schema inference. This is the default mode.
# EX: rescue: if you have: 1 file & 5 columns-> Next day 2 column added so file contains 7 columns.if we using rescue one column "rescue" will be created & in this column if schema/column change that schema/column data will come in dictionary format {id, name}.
# EX: addNewColumns: for this 2 new column will be added to the destination/final table

#autogpt
# option("cloudFiles.schemaEvolutionMode", "allowMissingColumns"): allowMissingColumns: Allow missing columns in the schema.
# option("cloudFiles.schemaEvolutionMode", "strict"): strict: Strict schema evolution. This is the default mode.
# option("cloudFiles.schemaEvolutionMode", "rescue"): rescue: Rescue files with schema inference. This is the default mode.

df = spark.readStream.format("cloudFiles").option("cloudFiles.format", "json").option("cloudFiles.schemaLocation", "/Volumes/data_dev/autoloader/bronze/destination/checkpoint/").option("cloudFiles.schemaEvolutionMode", "rescue").load("/Volumes/data_dev/autoloader/bronze/raw/")

In [0]:
#.trigger(): we can check every 5s/3s. As we are using Databricks free edition we use once = true. Means only one time it will run.  
df.writeStream.format("delta").outputMode("append").option("checkpointLocation", "/Volumes/data_dev/autoloader/bronze/destination/checkpoint/").trigger(once=True).start("/Volumes/data_dev/autoloader/bronze/destination/data/")

In [0]:
df1 = spark.read.format("delta").load("/Volumes/data_dev/autoloader/bronze/destination/data/")
display(df1)

-- 2nd run

Now will upload batch2.json to check rescue column working or not.  No need to create new code just run above code. you will see new data will be added.If you re rrun above sell also no problem. It will give the same result. Because stream already runs 

-- 3rd Run

or to check if it added new column in dictionary fomat or not using rescue. Now uploaded 3rd json files name batch3_schema_change.json here one new column is there payment method.After running the above shell as it is new column and data. This data will come in rescue in dictionary format.